```markdown
# 🧮 GSM8K Fine-tuning Pipeline - Improved Version
## Project: Training Chain-of-Thought Reasoning for Mathematical Problems
### Model: Qwen2.5-3B-Instruct
### Dataset: GSM8K (8.5K Mathematical Problems)
### Improvements: Error Handling, Validation, Better Prompts, Metrics
```

In [1]:

import os
import re
import json
from pathlib import Path
from typing import Dict, List, Optional
import warnings
warnings.filterwarnings('ignore')


# ============================================================
# Section 1: Installation and Imports
# ============================================================

def install_dependencies():
    """Install dependencies with error handling"""
    print("📦 Installing dependencies...")
    try:
        get_ipython().run_cell_magic('capture', '', '''
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9]{1,}\\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
        ''')
        print("✅ Installation successful")
    except Exception as e:
        print(f"❌ Installation error: {e}")
        raise

install_dependencies()

from google.colab import drive
from datasets import load_dataset
from unsloth import FastLanguageModel
from transformers import TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer
import torch
import numpy as np

print("✅ Libraries loaded successfully")

<>:23: SyntaxWarning: invalid escape sequence '\.'
<>:23: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipython-input-2751739416.py:23: SyntaxWarning: invalid escape sequence '\.'
  import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)


📦 نصب وابستگی‌ها...
✅ نصب موفق
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


✅ کتابخانه‌ها بارگذاری شدند


In [5]:
class Config:
    """Optimized settings to prevent overfitting"""

    # ============================================================
    # Paths
    # ============================================================
    DRIVE_MOUNT = "/content/drive"
    OUTPUT_DIR = "./qwen-gsm8k-checkpoints"
    FINAL_MODEL_DIR = "./qwen-gsm8k-final"
    DRIVE_SAVE_DIR = "/content/drive/MyDrive/qwen-gsm8k-math-reasoning"

    # ============================================================
    # Model Settings
    # ============================================================
    MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
    MAX_SEQ_LENGTH = 1536
    LOAD_IN_4BIT = True

    # ============================================================
    # LoRA Configuration (Overfitting Prevention)
    # ============================================================
    LORA_R = 8  # Reduced capacity to prevent overfitting
    LORA_ALPHA = 16  # alpha/r ratio = 2 (standard)
    LORA_DROPOUT = 0.1  # Reduced from 0.15 - better balance
    TARGET_MODULES = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
    LORA_BIAS = "none"
    TASK_TYPE = "CAUSAL_LM"

    # ============================================================
    # Training Hyperparameters (Tuned for Anti-Overfitting)
    # ============================================================
    BATCH_SIZE = 2
    GRADIENT_ACCUMULATION = 8  # Effective batch size = 16
    LEARNING_RATE = 5e-6  # 🔥 Significantly reduced - most critical change
    NUM_EPOCHS = 1  # 🔥 Reduced from 2 - based on previous logs
    WARMUP_RATIO = 0.05  # 🔥 Reduced from 0.15 - about 25-30 steps
    WEIGHT_DECAY = 0.01  # 🔥 Reduced from 0.1 - suitable for LoRA
    MAX_GRAD_NORM = 1.0

    # ============================================================
    # Scheduler & Optimizer
    # ============================================================
    LR_SCHEDULER_TYPE = "cosine"  # 🔥 Changed from cosine_with_restarts
    OPTIMIZER = "adamw_8bit"

    # ============================================================
    # Evaluation & Saving (Optimized)
    # ============================================================
    EVAL_STRATEGY = "steps"
    EVAL_STEPS = 50  # Frequent checks for early stopping
    SAVE_STRATEGY = "steps"
    SAVE_STEPS = 50
    SAVE_TOTAL_LIMIT = 2  # Only 2 best checkpoints
    LOGGING_STEPS = 10
    LOAD_BEST_MODEL_AT_END = True  # 🔥 Must be enabled
    METRIC_FOR_BEST_MODEL = "eval_loss"
    GREATER_IS_BETTER = False

    # ============================================================
    # Early Stopping (Critical for Preventing Overfitting)
    # ============================================================
    EARLY_STOPPING_PATIENCE = 3  # Stop after 3 worse evals
    EARLY_STOPPING_THRESHOLD = 0.001  # 🔥 Reduced from 0.01 - more sensitive

    # ============================================================
    # Memory Optimization
    # ============================================================
    GRADIENT_CHECKPOINTING = True
    BF16 = True
    FP16 = False

    # ============================================================
    # Data Loading & Augmentation
    # ============================================================
    DATALOADER_NUM_WORKERS = 2
    GROUP_BY_LENGTH = True
    SHUFFLE_DATASET = True
    MAX_SAMPLES = None
    NEFTUNE_NOISE_ALPHA = 2  # 🔥 Reduced from 5 - balanced data augmentation

    # ============================================================
    # Other Settings
    # ============================================================
    SEED = 3407
    NUM_TEST_SAMPLES = 10
    REPORT_TO = "none"
    DISABLE_TQDM = False

    # ============================================================
    # Debug & Analysis
    # ============================================================
    LOG_LEVEL = "info"
    LOGGING_FIRST_STEP = True
    EVALUATION_STRATEGY_DEBUG = False  # True for debug


# ============================================================
# Additional settings for use in code
# ============================================================
config = Config()



In [6]:

# ============================================================
# Section 3: Setup and System Check
# ============================================================

def setup_environment():
    """Setup environment and check resources"""
    print("\n" + "="*80)
    print("🔧 Setting up environment")
    print("="*80)

    # Mount Drive
    try:
        print("📂 Mounting Google Drive...")
        drive.mount(config.DRIVE_MOUNT)
        print("✅ Drive mounted successfully")
    except Exception as e:
        print(f"⚠️  Warning: Drive mount failed - {e}")

    # Check GPU
    if torch.cuda.is_available():
        gpu_stats = torch.cuda.get_device_properties(0)
        print(f"🎮 GPU: {gpu_stats.name}")
        print(f"💾 VRAM: {gpu_stats.total_memory / 1024**3:.2f} GB")
        print(f"✅ CUDA Version: {torch.version.cuda}")
    else:
        raise RuntimeError("❌ GPU not found! This code requires GPU.")

    # Create directories
    Path(config.OUTPUT_DIR).mkdir(exist_ok=True)
    Path(config.FINAL_MODEL_DIR).mkdir(exist_ok=True)

    print("✅ Environment is ready")

setup_environment()



🔧 راه‌اندازی محیط
📂 اتصال به Google Drive...
Mounted at /content/drive
✅ Drive متصل شد
🎮 GPU: Tesla T4
💾 VRAM: 14.74 GB
✅ CUDA Version: 12.6
✅ محیط آماده است


In [7]:

# ============================================================
# Section 4: Dataset Processing
# ============================================================

class GSM8KProcessor:
    """Class for processing and formatting GSM8K dataset"""

    @staticmethod
    def extract_final_answer(answer_text: str) -> Optional[str]:
        """Extract final number from answer"""
        match = re.search(r'####\s*(-?\d+(?:,\d+)*(?:\.\d+)?)', answer_text)
        if match:
            return match.group(1).replace(',', '')
        return None

    @staticmethod
    def format_prompt(example: Dict) -> Dict[str, str]:
        """Format to ChatML format with improved quality"""
        question = example['question'].strip()
        answer_full = example['answer'].strip()

        # Separate reasoning and answer
        if '####' in answer_full:
            reasoning, final = answer_full.split('####')
            reasoning = reasoning.strip()
            final = final.strip()
        else:
            reasoning = answer_full
            final = GSM8KProcessor.extract_final_answer(answer_full) or ""

        # Improved format with clearer instructions
        prompt = f"""<|im_start|>system
You are an expert math tutor. Solve problems step-by-step with clear reasoning.<|im_end|>
<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
{reasoning}

#### {final}<|im_end|>"""

        return {"text": prompt}

    @staticmethod
    def validate_sample(example: Dict) -> bool:
        """Validate sample correctness"""
        if not example.get('question') or not example.get('answer'):
            return False
        if '####' not in example['answer']:
            return False
        return True

def load_and_process_dataset():
    """Load and process dataset with validation"""
    print("\n" + "="*80)
    print("📚 بارگذاری دیتاست GSM8K")
    print("="*80)

    try:
        dataset = load_dataset("openai/gsm8k", "main", trust_remote_code=True)
    except Exception as e:
        print(f"❌ خطا در بارگذاری: {e}")
        raise

    print(f"✅ Train: {len(dataset['train'])} نمونه")
    print(f"✅ Test: {len(dataset['test'])} نمونه")

    # Display initial sample
    print("\n📋 نمونه از دیتاست:")
    print("-" * 80)
    sample = dataset['train'][0]
    print(f"❓ سوال: {sample['question'][:100]}...")
    print(f"💡 پاسخ: {sample['answer'][:100]}...")
    print("-" * 80)

    # Validation and filtering
    processor = GSM8KProcessor()

    print("\n🔄 پردازش و validation...")
    train_valid = sum(1 for ex in dataset['train'] if processor.validate_sample(ex))
    test_valid = sum(1 for ex in dataset['test'] if processor.validate_sample(ex))

    print(f"✅ Valid train samples: {train_valid}/{len(dataset['train'])}")
    print(f"✅ Valid test samples: {test_valid}/{len(dataset['test'])}")

    # Formatting
    train_dataset = dataset['train'].map(
        processor.format_prompt,
        remove_columns=dataset['train'].column_names,
        desc="Formatting train"
    )

    eval_dataset = dataset['test'].map(
        processor.format_prompt,
        remove_columns=dataset['test'].column_names,
        desc="Formatting test"
    )

    # Display formatted sample
    print("\n📋 نمونه فرمت‌شده:")
    print("-" * 80)
    print(train_dataset[0]['text'][:400])
    print("...")
    print("-" * 80)

    return train_dataset, eval_dataset

train_dataset, eval_dataset = load_and_process_dataset()


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'openai/gsm8k' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'openai/gsm8k' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.



📚 بارگذاری دیتاست GSM8K


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

✅ Train: 7473 نمونه
✅ Test: 1319 نمونه

📋 نمونه از دیتاست:
--------------------------------------------------------------------------------
❓ سوال: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How m...
💡 پاسخ: Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether...
--------------------------------------------------------------------------------

🔄 پردازش و validation...
✅ Valid train samples: 7473/7473
✅ Valid test samples: 1319/1319


Formatting train:   0%|          | 0/7473 [00:00<?, ? examples/s]

Formatting test:   0%|          | 0/1319 [00:00<?, ? examples/s]


📋 نمونه فرمت‌شده:
--------------------------------------------------------------------------------
<|im_start|>system
You are an expert math tutor. Solve problems step-by-step with clear reasoning.<|im_end|>
<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 cl
...
--------------------------------------------------------------------------------


In [8]:

# ============================================================
# Section 5: Model Loading and Configuration
# ============================================================

def load_model():
    """Load model with error handling"""
    print("\n" + "="*80)
    print("🤖 بارگذاری مدل")
    print("="*80)

    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=config.MODEL_NAME,
            max_seq_length=config.MAX_SEQ_LENGTH,
            dtype=None,
            load_in_4bit=config.LOAD_IN_4BIT,
        )
        print("✅ مدل بارگذاری شد")

        # Configure LoRA
        model = FastLanguageModel.get_peft_model(
            model,
            r=config.LORA_R,
            target_modules=config.TARGET_MODULES,
            lora_alpha=config.LORA_ALPHA,
            lora_dropout=config.LORA_DROPOUT,
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=config.SEED,
            use_rslora=False,
        )
        print("✅ LoRA تنظیم شد")

        # Display statistics
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())
        print(f"📊 Trainable: {trainable:,} ({100*trainable/total:.2f}%)")

        return model, tokenizer

    except Exception as e:
        print(f"❌ خطا در بارگذاری مدل: {e}")
        raise

model, tokenizer = load_model()



🤖 بارگذاری مدل
==((====))==  Unsloth 2025.12.5: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


✅ مدل بارگذاری شد


Unsloth 2025.12.5 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✅ LoRA تنظیم شد
📊 Trainable: 14,966,784 (0.87%)


In [9]:

# ============================================================
# Section 6: Training
# ============================================================

def create_training_args():
    """Create training configuration"""
    return TrainingArguments(
        output_dir=config.OUTPUT_DIR,

        # Batch
        per_device_train_batch_size=config.BATCH_SIZE,
        per_device_eval_batch_size=config.BATCH_SIZE,
        gradient_accumulation_steps=config.GRADIENT_ACCUMULATION,

        # Optimization
        learning_rate=config.LEARNING_RATE,
        warmup_ratio=config.WARMUP_RATIO,
        lr_scheduler_type="cosine",
        weight_decay=config.WEIGHT_DECAY,
        optim="adamw_8bit",
        max_grad_norm=config.MAX_GRAD_NORM,

        # Epochs
        num_train_epochs=config.NUM_EPOCHS,

        # Precision
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),

        # Logging & Saving
        logging_steps=config.LOGGING_STEPS,
        save_steps=config.SAVE_STEPS,
        save_total_limit=config.SAVE_TOTAL_LIMIT,
        eval_steps=config.EVAL_STEPS,

        # Evaluation
        eval_strategy="steps",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        # Memory
        gradient_checkpointing=True,

        # Other
        report_to="none",
        seed=config.SEED,
        dataloader_num_workers=2,
        remove_unused_columns=False,
    )

def train_model():
    """Train model with monitoring"""
    print("\n" + "="*80)
    print("🚀 شروع Fine-tuning")
    print("="*80)

    training_args = create_training_args()

    effective_batch = config.BATCH_SIZE * config.GRADIENT_ACCUMULATION
    total_steps = (len(train_dataset) * config.NUM_EPOCHS) // effective_batch

    print(f"📊 تنظیمات:")
    print(f"   • Samples: {len(train_dataset)} train, {len(eval_dataset)} eval")
    print(f"   • Batch size: {effective_batch} (effective)")
    print(f"   • Epochs: {config.NUM_EPOCHS}")
    print(f"   • Total steps: ~{total_steps}")
    print(f"   • Learning rate: {config.LEARNING_RATE}")
    print("-" * 80)

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        dataset_text_field="text",
        max_seq_length=config.MAX_SEQ_LENGTH,
        args=training_args,
        packing=False,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    try:
        trainer.train()
        print("\n✅ آموزش با موفقیت تکمیل شد!")
        return trainer
    except Exception as e:
        print(f"\n❌ خطا در آموزش: {e}")
        raise

trainer = train_model()



🚀 شروع Fine-tuning
📊 تنظیمات:
   • Samples: 7473 train, 1319 eval
   • Batch size: 16 (effective)
   • Epochs: 1
   • Total steps: ~467
   • Learning rate: 5e-06
--------------------------------------------------------------------------------


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/7473 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1319 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,473 | Num Epochs = 1 | Total steps = 468
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 14,966,784 of 3,100,905,472 (0.48% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
50,1.305800,1.711238
100,0.817200,1.252656
150,0.609200,1.075422
200,0.533200,1.025318
250,0.489100,0.997728
300,0.474000,0.985756
350,0.461500,0.976254
400,0.444800,0.972529


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Step,Training Loss,Validation Loss
50,1.305800,1.711238
100,0.817200,1.252656
150,0.609200,1.075422
200,0.533200,1.025318
250,0.489100,0.997728
300,0.474000,0.985756
350,0.461500,0.976254
400,0.444800,0.972529
450,0.456000,0.971260



✅ آموزش با موفقیت تکمیل شد!


In [10]:

# ============================================================
# Section 7: Model Saving
# ============================================================

def save_model():
    """Save model to different paths"""
    print("\n" + "="*80)
    print("💾 ذخیره مدل")
    print("="*80)

    # Local save
    try:
        model.save_pretrained(config.FINAL_MODEL_DIR)
        tokenizer.save_pretrained(config.FINAL_MODEL_DIR)
        print(f"✅ Local: {config.FINAL_MODEL_DIR}")
    except Exception as e:
        print(f"⚠️  Local save failed: {e}")

    # Drive save
    try:
        Path(config.DRIVE_SAVE_DIR).mkdir(parents=True, exist_ok=True)
        model.save_pretrained(config.DRIVE_SAVE_DIR)
        tokenizer.save_pretrained(config.DRIVE_SAVE_DIR)
        print(f"✅ Drive: {config.DRIVE_SAVE_DIR}")

        # Save metadata
        metadata = {
            "model": config.MODEL_NAME,
            "epochs": config.NUM_EPOCHS,
            "batch_size": config.BATCH_SIZE * config.GRADIENT_ACCUMULATION,
            "learning_rate": config.LEARNING_RATE,
            "train_samples": len(train_dataset),
        }
        with open(f"{config.DRIVE_SAVE_DIR}/training_info.json", "w") as f:
            json.dump(metadata, f, indent=2)
        print("✅ Metadata ذخیره شد")

    except Exception as e:
        print(f"⚠️  Drive save failed: {e}")

save_model()



💾 ذخیره مدل
✅ Local: ./qwen-gsm8k-final
✅ Drive: /content/drive/MyDrive/qwen-gsm8k-math-reasoning
✅ Metadata ذخیره شد


In [11]:
def test_model():
    """Evaluate GSM8K-style reasoning after fine-tuning"""
    import torch
    from unsloth import FastLanguageModel

    print("\n" + "=" * 80)
    print("🧪 GSM8K Fine-Tuning Evaluation")
    print("=" * 80)

    # Put model in inference mode
    FastLanguageModel.for_inference(model)
    model.eval()

    test_questions = [
        "Sarah has 24 apples. She gives one third of them to her friend and then buys 4 more apples. How many apples does she have now?",
        "Mike had 18 pencils. After giving some away, he has two thirds of his pencils left. Then he buys 6 more pencils. How many pencils does he have in total?",
        "A shop has 40 chocolates. First, 25% of the chocolates are sold. Then 8 more chocolates are added to the shop. After that, half of the remaining chocolates are sold. How many chocolates remain?",
        "A bottle contains 2.5 liters of water. 40% of the water is used, and then 0.6 liters of water are added. How much water is in the bottle now?",
        "Mary has 3 boxes of balls. Each box contains 12 balls. She loses 5 balls and then buys 7 new balls. How many balls does she have in total?",
        "After giving away 8 candies, John has 20 candies left. How many candies did John have originally?",
        "A car travels at 60 miles per hour for 2 hours and then at 40 miles per hour for 1 hour. How many miles does the car travel in total?",
        "A baker makes 5 trays of cookies. Each tray has 16 cookies. He sells 38 cookies. How many cookies remain?",
        "Emma spent one fourth of her money on a book. She then earned $15 and now has $45. How much money did Emma have at the start?",
        "There are 30 students in a class. 40% of them are girls. If 3 more girls join the class, how many boys are in the class now?"
    ]

    for i, question in enumerate(test_questions, 1):
        print("\n" + "=" * 70)
        print(f"Test {i}/{len(test_questions)}")
        print("=" * 70)

        prompt = f"""<|im_start|>system
You are an expert math tutor. Solve the problem step-by-step and clearly state the final answer.<|im_end|>
<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
"""

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.0,          # deterministic for fair comparison
                top_p=1.0,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

        # Extract assistant response
        if "<|im_start|>assistant" in decoded:
            answer = decoded.split("<|im_start|>assistant")[-1]
            answer = answer.split("<|im_end|>")[0].strip()
        else:
            answer = decoded.strip()

        print(f"❓ Question:\n{question}\n")
        print(f"💡 Model Answer:\n{answer}")

    print("\n" + "=" * 80)
    print("✅ Evaluation completed")
    print("=" * 80)


# Run evaluation
test_model()



🧪 GSM8K Fine-Tuning Evaluation

Test 1/10
❓ Question:
Sarah has 24 apples. She gives one third of them to her friend and then buys 4 more apples. How many apples does she have now?

💡 Model Answer:
Step 1: Calculate how many apples Sarah gives to her friend.
One third of 24 is 24 / 3 = <<24/3=8>>8

Step 2: Subtract the number of apples given away from the original amount.
24 - 8 = <<24-8=16>>16

Step 3: Add the 4 new apples Sarah bought.
16 + 4 = <<16+4=20>>20

Final Answer: Sarah now has 20 apples.

Test 2/10
❓ Question:
Mike had 18 pencils. After giving some away, he has two thirds of his pencils left. Then he buys 6 more pencils. How many pencils does he have in total?

💡 Model Answer:
Step 1: Mike originally had 18 pencils.
Step 2: He gave away some pencils and is left with 2/3 of his original amount.
Step 3: Calculate how many pencils he has left after giving some away:
   2/3 * 18 = <<2/3*18=12>>12 pencils
Step 4: Mike then buys 6 more pencils.
Step 5: Add the 6 new pencils to t

In [12]:
def test_base_model_fresh():
    """Load and test PURE base model from scratch"""
    import torch
    from unsloth import FastLanguageModel
    import gc

    print("\n" + "=" * 80)
    print("🧪 Pure Base Model Evaluation (Fresh Load - NO LoRA)")
    print("=" * 80)

    # Base model specifications - replace with your model specs
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"  # یا هر مدلی که استفاده کردی
    max_seq_length = 2048
    dtype = None
    load_in_4bit = True

    print(f"\n📥 Loading PURE base model (without any LoRA): {model_name}")
    print("⏳ Please wait, loading fresh model...")

    # Load base model without any LoRA
    base_model, base_tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
        # Important: Do not add any adapters
    )

    # Inference mode only
    FastLanguageModel.for_inference(base_model)
    base_model.eval()

    print("✅ Pure base model loaded successfully!\n")

    test_questions = [
        "Sarah has 24 apples. She gives one third of them to her friend and then buys 4 more apples. How many apples does she have now?",
        "Mike had 18 pencils. After giving some away, he has two thirds of his pencils left. Then he buys 6 more pencils. How many pencils does he have in total?",
        "A shop has 40 chocolates. First, 25% of the chocolates are sold. Then 8 more chocolates are added to the shop. After that, half of the remaining chocolates are sold. How many chocolates remain?",
        "A bottle contains 2.5 liters of water. 40% of the water is used, and then 0.6 liters of water are added. How much water is in the bottle now?",
        "Mary has 3 boxes of balls. Each box contains 12 balls. She loses 5 balls and then buys 7 new balls. How many balls does she have in total?",
        "After giving away 8 candies, John has 20 candies left. How many candies did John have originally?",
        "A car travels at 60 miles per hour for 2 hours and then at 40 miles per hour for 1 hour. How many miles does the car travel in total?",
        "A baker makes 5 trays of cookies. Each tray has 16 cookies. He sells 38 cookies. How many cookies remain?",
        "Emma spent one fourth of her money on a book. She then earned $15 and now has $45. How much money did Emma have at the start?",
        "There are 30 students in a class. 40% of them are girls. If 3 more girls join the class, how many boys are in the class now?"
    ]

    for i, question in enumerate(test_questions, 1):
        print("\n" + "=" * 70)
        print(f"Test {i}/{len(test_questions)} - 🔵 PURE BASE MODEL")
        print("=" * 70)

        prompt = f"""<|im_start|>system
You are an expert math tutor. Solve the problem step-by-step and clearly state the final answer.<|im_end|>
<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
"""

        inputs = base_tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = base_model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.0,
                top_p=1.0,
                do_sample=False,
                pad_token_id=base_tokenizer.eos_token_id,
            )

        decoded = base_tokenizer.decode(outputs[0], skip_special_tokens=False)

        # Extract assistant response
        if "<|im_start|>assistant" in decoded:
            answer = decoded.split("<|im_start|>assistant")[-1]
            answer = answer.split("<|im_end|>")[0].strip()
        else:
            answer = decoded.strip()

        print(f"❓ Question:\n{question}\n")
        print(f"💡 Pure Base Model Answer:\n{answer}")

    print("\n" + "=" * 80)
    print("✅ Pure Base Model Evaluation Completed")
    print("=" * 80)

    # Clear model from memory
    del base_model
    del base_tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("\n🧹 Base model cleared from memory")

# Run test
test_base_model_fresh()


🧪 Pure Base Model Evaluation (Fresh Load - NO LoRA)

📥 Loading PURE base model (without any LoRA): unsloth/Qwen2.5-3B-Instruct-bnb-4bit
⏳ Please wait, loading fresh model...
==((====))==  Unsloth 2025.12.5: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Pure base model loaded successfully!


Test 1/10 - 🔵 PURE BASE MODEL
❓ Question:
Sarah has 24 apples. She gives one third of them to her friend and then buys 4 more apples. How many apples does she have now?

💡 Pure Base Model Answer:
Let's break down the problem step by step:

1. Sarah starts with 24 apples.
2. She gives away one third of her apples to her friend